[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/morganmcg1/deep-research-bot/blob/main/notebooks/01_simple_agent.ipynb)


# Introduction to Building LLM Agents with Tools and Tracing

<!--- @wandbcode{fc-london-workshop-2025} -->

This script walks through the process of building a simple LLM-powered agent that can use tools (functions) to answer questions. We'll cover:
1. Making basic LLM calls.
2. Introducing Weave for tracing and observability.
3. Defining tools for the LLM (manually and automatically).
4. Implementing a basic agentic loop.
5. Structuring the agent using Python classes.
6. Running the agent on a multi-step task.

**Prerequisites:**
Make sure you have the necessary libraries installed:
```bash
!uv pip install -qqq git+https://github.com/morganmcg1/deep-research-bot.git
```

Note: I had to fix dependencies for weave:
```
    "pydantic>=2.0,<3.0",
    "weave>=0.50.0",
```


In [1]:
!uv pip install -qqq git+https://github.com/morganmcg1/deep-research-bot.git

In [2]:
# Global Configuration & Setup
import os
import inspect
import json
import weave # Must import weave before litellm for auto-patching
import openai
from enum import Enum
from pydantic import BaseModel, Field
from rich.panel import Panel
from rich.markdown import Markdown
from rich.console import Console as RichConsole
from exa_py import Exa
from typing import Any, Callable, get_type_hints

### Add your API Keys
We'll need a wandb api key and an exa api key

In [3]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [4]:
# or

os.environ["WANDB_API_KEY"] = "4d3ab3d90d56fefecd0c6a8c8242a48141814acd"
os.environ["EXA_API_KEY"] = "ac888a64-d2a2-4d47-9476-ceb71058344f"

# or colab secrets: 

# from google.colab import userdata
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# os.environ["EXA_API_KEY"] = userdata.get("EXA_API_KEY")

## Model Settings
Define a model to use, as we are going to use tool calling you need a capable model like `Kimi-K2` or `GLM`

In [ ]:
class Console(RichConsole):
    def md(self, text): 
        return self.print(Markdown(text))

console = Console()

MODEL_SMALL = "Qwen/Qwen3-235B-A22B-Instruct-2507"
MODEL_MEDIUM = "zai-org/GLM-4.5"
MODEL_LARGE = "moonshotai/Kimi-K2-Instruct"

WANDB_ENTITY = "ramsey-wise"
WANDB_PROJECT = "london-workshop-2025"

oai_client = openai.OpenAI(
    base_url='https://api.inference.wandb.ai/v1',
    api_key=os.getenv("WANDB_API_KEY"),
    project=f"{WANDB_ENTITY}/{WANDB_PROJECT}")
exa_client = Exa(api_key=os.getenv("EXA_API_KEY"))

Let's log to [W&B Weave](https://weave-docs.wandb.ai/). Weights & Biases (W&B) Weave is a framework for tracking, experimenting with, evaluating, deploying, and improving LLM-based applications. Designed for flexibility and scalability, Weave supports every stage of your LLM application development workflow:

- Tracing & Monitoring: Track LLM calls and application logic to debug and analyze production systems.
- Systematic Iteration: Refine and iterate on prompts, datasets, and models.
- Experimentation: Experiment with different models and prompts in the LLM Playground.
- Evaluation: Use custom or pre-built scorers alongside our comparison tools to systematically assess and enhance application performance.
- Guardrails: Protect your application with pre- and post-safeguards for content moderation, prompt safety, and more.

In [ ]:

# Initialize a Weave project. Traces will be sent here.
# You can view them in the Weave UI (usually runs locally).
weave.init(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

weave: wandb version 0.23.1 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: weave version 0.52.22 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: ramsey-wise.
weave: View Weave data at https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/weave


## 1. Basic LLM Call with OpenAI SDK

Let's start with a simple call to the LLM using/

![](images/01_trace.png)

In [7]:
# Define a simple message list (conversation history)
messages = [{"role": "user", "content": "Hello, LLM! How does an AI agent work?"}]

# Make the call
response = resp = oai_client.chat.completions.create(
    model = MODEL_SMALL,
    messages=messages,
    max_tokens=400,
)
# Print the response content
assistant_response = response.choices[0].message.content
console.md(f"LLM Response:\\n{assistant_response}")

# Click on the 🍩 linke below to see the trace in Weave 👇

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e21-d343-7dcd-9bca-60b3bbea47c7


LLM Response:\nHello! Great question! 😊                                                                           

An AI agent is a software system that perceives its environment, makes decisions, and takes actions to achieve     
specific goals. Think of it like a digital "assistant" that can observe, think, and act—sometimes even learning    
from experience.                                                                                                   

Here’s a breakdown of how an AI agent works:                                                                       

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                      1. Perception (Sensing the Environment)                                      

The agent gathers information from its environment using inputs such as:                                           

 • Text (e.g., user messages)                                                                                      
 • Sensors (in robotics)                                                                                           
 • Databases or APIs                                                                                               
 • Images, audio, etc.                                                                                             

👉 Example: A chatbot reads your message: "What's the weather today?"                                              

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                   2. Processing & Decision-Making (The "Brain")                                   

This is where the AI analyzes the input and decides what to do. It may use:                                        

 • Rules or logic (if-then statements)                                                                             
 • Machine learning models (like neural networks)                                                                  
 • Large language models (LLMs) (like me!) to understand and generate responses                                    
 • Planning algorithms (to map out steps toward a goal)                                                            

👉 Example: The agent understands your question is about weather and determines it needs to fetch current weather  
data.                                                                                                              

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                         3. Action (Responding or Acting)                                          

The agent performs an action, such as:                                                                             

 • Sending a message                                                                                               
 • Moving a robot arm                                                                                              
 • Updating a database                                                                                             
 • Making a recommendation                                                                                         

👉 Example: The agent queries a weather API and replies: "It's 22°C and sunny in your area."                       

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                 4. Learning & Adaptation (Optional but Powerful)                                  

Some agents improve over time using:                                                                               

 • Reinforcement learning (learning from rewards/punishments)                                                      
 • Feedback loops (e.g., user ratings)     

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e21-e850-710a-a456-b2c66573d860
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e21-fa15-79b2-a336-c6581ffa57dc
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e22-f9b7-7c1c-aec4-46aaa4eb0c19
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e24-7db3-7517-8933-96d6a3f88ad9
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e24-ac81-7c4b-92cb-8f6325307fba
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e25-1fd1-7089-a4f8-9a4f8fa86de5


Because we imported `weave` and called `weave.init()`, the OpenAI SDK call above was automatically traced. You can open your Weave dashboard and see the trace, including the input messages, output response, latency, model used, etc. This is invaluable for debugging and monitoring.

In [8]:
# most of the time you would want to define your own operations to trace, for instance to call the model.
# You just need to add the @weave.op decorator to the function and it will be traced.

@weave.op
def call_model(model_name: str, messages: list[dict[str, Any]], **kwargs) -> str:
    "Call a model with the given messages and kwargs."
    response = oai_client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=400,
        **kwargs
    )

    return response.choices[0].message

response = call_model(model_name=MODEL_SMALL, messages=messages)
console.md(response.content)

Hello! Great question! 😊                                                                                          

An AI agent is a software system that perceives its environment, makes decisions, and takes actions to achieve     
specific goals. Think of it like a digital "assistant" that can think and act autonomously, to some degree.        

Here’s a simple breakdown of how an AI agent works:                                                                

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                   1. Perception                                                   

The agent gathers information from its environment using sensors or inputs. This could be:                         

 • Text input (like your message to me)                                                                            
 • Sensor data (e.g., camera, microphone)                                                                          
 • Data from databases or APIs                                                                                     

➡️ Example: A self-driving car uses cameras and radar to "see" the road.                                            

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                          2. Processing & Decision-Making                                          

The agent uses AI models (like machine learning or large language models) to:                                      

 • Understand the input                                                                                            
 • Reason about possible actions                                                                                   
 • Decide what to do next                                                                                          

This step often involves:                                                                                          

 • Pattern recognition                                                                                             
 • Planning                                                                                                        
 • Learning from past experiences                                                                                  

➡️ Example: A chatbot like me reads your question, understands the intent, and figures out a helpful response.      

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                     3. Action                                                     

The agent performs an action based on its decision. This could be:                                                 

 • Sending a message                                                                                               
 • Moving a robot arm                                                                                              
 • Adjusting a thermostat                                                                                          
 • Recommending a product                                                                                          

➡️ Example: I type this reply and send it to you!                                                                   

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                        4. Learning (Optional but Powerful)                                        

Many AI agents improve over time by learning from feedback or experience:                                          

 • Reinforcement learning: learns from rewards/punishments                                                         
 • Supervised learning: learns from

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e23-243e-79c0-97c4-574df43cba53
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e27-1aaf-763b-9650-daa165f659aa
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e27-f1a8-7e99-9e9c-6b8f3c39538a
weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e2f-f632-7628-b504-a7e15b82f406
INFO:weave.trace.weave_client:🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e2f-f632-7628-b504-a7e15b82f406


![](images/02_nested_trace.png)


## 2. Introducing Tool Calling

Agents become much more powerful when they can use **tools** – external functions or APIs – to get information or perform actions beyond the LLM's internal knowledge. To allow an LLM to use a tool, we need to provide it with a description (schema) of the tool, including its name, purpose, and expected arguments.

Check the Mistral docs for function calling: https://platform.openai.com/docs/guides/function-calling

![](images/function-calling-diagram-steps.png)

First, let's define a simple Python function we want the LLM to be able to call. We add `@weave.op` to trace when this function actually gets executed.


In [9]:
@weave.op 
def add_numbers(a: int, b: int) -> int:
    """Use this tool to add numbers.
    Args:
        a: The first number.
        b: The second number.
    """
    return a + b

In [ ]:
add_numbers(1, 2)

3

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e21-fa0f-7f6b-9b4e-5977be268235


weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e24-a688-7cbd-9e51-60db5cb1ff87


In [11]:
# this doesn't work...
call_model(model_name=MODEL_SMALL, messages=messages, tools=[add_numbers])

TypeError: Object of type function is not JSON serializable

> We need to manually create the JSON schema describing this tool in a format that models *Mistral* understand.

In [12]:
# Manually define the tool schema
tool_add_numbers_schema = {
    "type": "function",
    "function": {
        "name": "add_numbers",
        "description": "Adds two numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "The first number."
                },
                "b": {
                    "type": "integer",
                    "description": "The second number."
                }
            },
            "required": ["a", "b"]
        }
    }
}

Now, we make an LLM call, passing the `tools` parameter with our schema. We ask a question that should trigger the tool.

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant use tools to answer questions."},
    {"role": "user", "content": "My lucky numbers are 77 and 11. What is their sum?"}]
response = call_model(model_name=MODEL_SMALL, messages=messages, tools=[tool_add_numbers_schema])
console.print(response)

ChatCompletionMessage(
    content=None,
    refusal=None,
    role='assistant',
    annotations=None,
    audio=None,
    function_call=None,
    tool_calls=[
        ChatCompletionMessageFunctionToolCall(
            id='chatcmpl-tool-cddf8995fcf6496da3d45dd385f3da27',
            function=Function(arguments='{"a": 77, "b": 11}', name='add_numbers'),
            type='function'
        )
    ],
    reasoning_content=None
)

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e25-7781-7a57-a0c7-d5bcae030d9e


## Manual Tool Call
The LLM's response might contain a request to call our tool (`response.choices[0].message.tool_calls`) or it might respond directly (`response.choices[0].message.content`). If it requests a tool call, we need to:

1. Parse the arguments it provides.
2. Execute our actual Python function (`add_numbers`) with those arguments.
3. (In a real agent loop) Send the result back to the LLM in a new message with `role="tool"`.

Let's manually call the tools in the response.

In [14]:
if response.tool_calls:
    console.print("LLM requested a tool call:")
    for tool_call in response.tool_calls:
        function_name = tool_call.function.name
        function_args_str = tool_call.function.arguments
        function_args = json.loads(function_args_str)
        console.print(f"  - Tool: {function_name}, Args: {function_args_str}")
        if function_name == "add_numbers":
            tool_result_content = add_numbers(**function_args)

console.print(f"Final Result: {tool_result_content}")

LLM requested a tool call:

- Tool: add_numbers, Args: {"a": 77, "b": 11}

Final Result: 88

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e23-012b-70bb-9f44-58ba6212e8f6


We need to add the tool call result to the messages (there is actually 2 messages to add)
- the response from the assistant that decided to call the tool
- the tool output

In [15]:
messages.append(response.model_dump())

In [16]:
messages.append({
    "tool_call_id": tool_call.id,
    "role": "tool",
    "name": function_name,
    "content": str(tool_result_content)
})

In [17]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant use tools to answer questions.'},
 {'role': 'user',
  'content': 'My lucky numbers are 77 and 11. What is their sum?'},
 {'content': None,
  'refusal': None,
  'role': 'assistant',
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'chatcmpl-tool-cddf8995fcf6496da3d45dd385f3da27',
    'function': {'arguments': '{"a": 77, "b": 11}', 'name': 'add_numbers'},
    'type': 'function'}],
  'reasoning_content': None},
 {'tool_call_id': 'chatcmpl-tool-cddf8995fcf6496da3d45dd385f3da27',
  'role': 'tool',
  'name': 'add_numbers',
  'content': '88'}]

You should have a sequence of messages like this:


In [18]:
[m["role"] for m in messages]

['system', 'user', 'assistant', 'tool']

Now call the model again with the new messages and it will use the tool call result to answer the question

In [ ]:
final_response = call_model(model_name=MODEL_SMALL, messages=messages)
console.print(final_response.content)

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e23-0776-7f52-b55c-6c344654a6ea


The sum of your lucky numbers, 77 and 11, is **88**.

weave: 🍩 https://wandb.ai/ramsey-wise-sevdesk/london-workshop-2025/r/call/019b9e24-3aba-75ee-9fe6-0daa2358a0da


## 3. Simplifying Tool Definition with a Processor Function

Manually writing JSON schemas is tedious and error-prone. We can automate this by inspecting our Python function's signature, type hints, and docstring.

First, let's define a helper function (`generate_tool_schema`) that takes a Python function and generates the schema.


In [20]:
def generate_tool_schema(func: Callable) -> dict:
    """Given a Python function, generate a tool-compatible JSON schema.
    Handles basic types and Enums. Assumes docstrings are formatted for arg descriptions.
    """
    signature = inspect.signature(func)
    parameters = signature.parameters
    type_hints = get_type_hints(func)

    schema = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": inspect.getdoc(func).split("\\n")[0] if inspect.getdoc(func) else "",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    }

    docstring = inspect.getdoc(func)
    param_descriptions = {}
    if docstring:
        args_section = False
        current_param = None
        for line in docstring.split('\\n'):
            line_stripped = line.strip()
            if line_stripped.lower().startswith(("args:", "arguments:", "parameters:")):
                args_section = True
                continue
            if args_section:
                if ":" in line_stripped:
                    param_name, desc = line_stripped.split(":", 1)
                    param_descriptions[param_name.strip()] = desc.strip()
                elif line_stripped and not line_stripped.startswith(" "): # Heuristic: end of args section
                     args_section = False

    for name, param in parameters.items():
        is_required = param.default == inspect.Parameter.empty
        param_type = type_hints.get(name, Any)
        json_type = "string"
        param_schema = {}

        # Basic type mapping
        if param_type == str: json_type = "string"
        elif param_type == int: json_type = "integer"
        elif param_type == float: json_type = "number"
        elif param_type == bool: json_type = "boolean"
        elif hasattr(param_type, '__origin__') and param_type.__origin__ is list: # Handle list[type]
             item_type = param_type.__args__[0] if param_type.__args__ else Any
             if item_type == str: param_schema = {"type": "array", "items": {"type": "string"}}
             elif item_type == int: param_schema = {"type": "array", "items": {"type": "integer"}}
             # Add more list item types if needed
             else: param_schema = {"type": "array", "items": {"type": "string"}} # Default list item type
        elif hasattr(param_type, "__members__") and issubclass(param_type, Enum): # Handle Enum
             json_type = "string"
             param_schema["enum"] = [e.value for e in param_type]

        if not param_schema: # If not set by list or Enum
            param_schema["type"] = json_type

        param_schema["description"] = param_descriptions.get(name, "")

        if param.default != inspect.Parameter.empty and param.default is not None:
             param_schema["default"] = param.default # Note: OpenAI schema doesn't officially use default, but useful metadata

        schema["function"]["parameters"]["properties"][name] = param_schema
        if is_required:
            schema["function"]["parameters"]["required"].append(name)
    return schema

Now we can use this function to automatically generate the schema for our tool.

In [21]:
tool_schema = generate_tool_schema(add_numbers)
console.print(tool_schema)

{
    'type': 'function',
    'function': {
        'name': 'add_numbers',
        'description': 'Use this tool to add numbers.\nArgs:\n    a: The first number.\n    b: The second number.',
        'parameters': {
            'type': 'object',
            'properties': {
                'a': {'type': 'integer', 'description': ''},
                'b': {'type': 'integer', 'description': ''}
            },
            'required': ['a', 'b']
        }
    }
}

Now, we define a `function_tool` "processor". This isn't a decorator in the `@` syntax sense here, but a function that we call *after* defining our tool function. It uses `generate_tool_schema` to attach the schema to the function object itself.


In [22]:
def function_tool(func: Callable) -> Callable:
    """Attaches a tool schema to the function and marks it as a tool.
    Call this *after* defining your function: my_func = function_tool(my_func)
    """
    try:
        func.tool_schema = generate_tool_schema(func)
        func.is_tool = True # Mark it as a tool
    except Exception as e:
        console.print(f"Error processing tool {func.__name__}: {e}")
        # Optionally raise or mark as failed
        func.tool_schema = None
        func.is_tool = False
    return func

We can use this function to automatically generate the schema for our tool, as a decorator or after the function is defined.

In [23]:
add_numbers = function_tool(add_numbers)
console.print(add_numbers.tool_schema)

{
    'type': 'function',
    'function': {
        'name': 'add_numbers',
        'description': 'Use this tool to add numbers.\nArgs:\n    a: The first number.\n    b: The second number.',
        'parameters': {
            'type': 'object',
            'properties': {
                'a': {'type': 'integer', 'description': ''},
                'b': {'type': 'integer', 'description': ''}
            },
            'required': ['a', 'b']
        }
    }
}

In [24]:
add_numbers.tool_schema

{'type': 'function',
 'function': {'name': 'add_numbers',
  'description': 'Use this tool to add numbers.\nArgs:\n    a: The first number.\n    b: The second number.',
  'parameters': {'type': 'object',
   'properties': {'a': {'type': 'integer', 'description': ''},
    'b': {'type': 'integer', 'description': ''}},
   'required': ['a', 'b']}}}

and call the tool =)

In [25]:
add_numbers(1, 2)

3

### 3.1 Real Example using an API based tool

We are going to use the [EXA search API](https://docs.exa.ai/reference/getting-started).
- How does [EXA search works](https://docs.exa.ai/reference/how-exa-search-works#how-exa-search-works)
- Using exa search [as tool calling](https://docs.exa.ai/reference/openai-tool-calling)

In [26]:
query = "Recipes for cooking seabass?"

search_res = exa_client.search_and_contents(query=query, type='auto', num_results=4)

In [27]:
console.print(search_res)

SearchResponse(
    results=[
        Result(
            url='https://www.facebook.com/chefbillyparisi/posts/the-best-way-to-cook-chilean-sea-bass-that-takes-it
-to-the-next-level/1239156604237561/',
            id='https://www.facebook.com/chefbillyparisi/posts/the-best-way-to-cook-chilean-sea-bass-that-takes-it-
to-the-next-level/1239156604237561/',
            title='The BEST way to Cook Chilean Sea Bass that takes it ...',
            score=None,
            published_date='2025-04-08T15:04:24.542Z',
            author='',
            image=None,
            favicon='https://static.xx.fbcdn.net/rsrc.php/yx/r/e9sqr8WnkCf.ico',
            subpages=None,
            extras=None,
            text='You’re Temporarily Blocked It looks like you were misusing this feature by going too fast. You’ve
been temporarily blocked from using it. Back',
            highlights=None,
            highlight_scores=None,
            summary=None
        ),
        Result(
            url='https://www.bowlofdelicious.com/pan-fried-sea-bass-with-lemon-garlic-herb-sauce/',
            id='https://www.bowlofdelicious.com/pan-fried-sea-bass-with-lemon-garlic-herb-sauce/',
            title='Pan Fried Sea Bass with Lemon Garlic Herb Sauce',
            score=None,
            published_date='2017-08-16T00:00:00.000Z',
            author='Elizabeth Lindemann',
            image='https://www.bowlofdelicious.com/wp-content/uploads/2017/08/Pan-Fried-Sea-Bass-with-Lemon-Garlic-
Herb-Sauce-square.jpg',
            favicon='https://www.bowlofdelicious.com/wp-content/uploads/2021/09/cropped-favicon-32x32.png',
            subpages=None,
            extras=None,
            text="- [Skip to primary 
navigation](https://www.bowlofdelicious.com/www.bowlofdelicious.com#genesis-nav-primary)\n- [Skip to main 
content](https://www.bowlofdelicious.com/www.bowlofdelicious.com#genesis-content)\n- [Skip to primary 
sidebar](https://www.bowlofdelicious.com/www.bowlofdelicious.com#genesis-sidebar-primary)\n\nSearch\n\n- 
[About](https://www.bowlofdelicious.com/about/)\n- [Recipes](https://www.bowlofdelicious.com/recipes/)\n- 
[Contact](https://www.bowlofdelicious.com/contact/)\n- 
[Subscribe](https://www.bowlofdelicious.com/www.bowlofdelicious.com#subscribe)\n\n[×](https://www.bowlofdelicious.c
om/www.bowlofdelicious.com)\n\n[Jump to Recipe](https://www.bowlofdelicious.com/www.bowlofdelicious.com#recipe) \\-
[Print](https://www.bowlofdelicious.com/wprm_print/pan-fried-sea-bass-with-lemon-garlic-herb-sauce)\n\n4.98 from 
655 votes\n\n_This Pan Fried Sea Bass with Lemon Garlic Herb Sauce is a 20 minute, one-pan, budget-friendly 
Mediterranean recipe that’s perfect on busy weeknights._\n\nThis Pan Fried Sea Bass recipe is a wonderful way to 
eat a mild, affordable\\* fish, and it’s packed with flavor from fresh oregano, thyme, and parsley. It’s especially
great if you have an abundance of fresh herbs in your garden you need to use up! _(raises hand)_\n\n\\\\* _Sea bass
can vary greatly in price depending on variety, location, and if it’s in season. Barramundi is a reliably 
affordable option and is what I used._\n\nOriginally, I was planning on doing a lemon caper sauce, similar to 
chicken piccata. But lo and behold, when I arrived at my refrigerator, I realized I was out of capers.\n\nAfter 
some quick thinking and wandering around outside, I decided on a lemony, garlicky, fresh herb sauce instead.\n\nSO 
happy I was out of capers. This turned out wonderfully!\n\n## How to Pan-Fry Sea Bass\n\nIf you’ve ever made 
chicken piccata before, the process for this is very similar. **First, pat the fish dry.** It’s important that most
of the moisture is removed from the fish, so it gets a nice brown color on the outside.\n\nNext\xa0**dredge the 
fish** in a mixture of all-purpose flour, kosher salt, and black pepper. Dredging simply means coating it in the 
flour mixture, and shaking off any excess. You should have a nice, thin coat of flour, salt, and pepper that covers
every s

Let's explore the payload

In [28]:
console.md("\n-------------------\n".join(result.text for result in search_res.results))

You’re Temporarily Blocked It looks like you were misusing this feature by going too fast. You’ve been temporarily 
                                            blocked from using it. Back                                            

 • ]8;id=654250;https://www.bowlofdelicious.com/www.bowlofdelicious.com#genesis-nav-primary\Skip to primary navigation]8;;\                                                                                      
 • ]8;id=682139;https://www.bowlofdelicious.com/www.bowlofdelicious.com#genesis-content\Skip to main content]8;;\                                                                                            
 • ]8;id=601741;https://www.bowlofdelicious.com/www.bowlofdelicious.com#genesis-sidebar-primary\Skip to primary sidebar]8;;\                                                                                         

Search                                                                                                             

 • ]8;id=367024;https://www.bowlofdelicious.com/about/\About]8;;\                                                                                                           
 • ]8;id=330869;https://www.bowlofdelicious.com/recipes/\Recipes]8;;\                                                                                                         
 • ]8;id=860796;https://www.bowlofdelicious.com/contact/\Contact]8;;\                                                                                                         
 • ]8;id=57885;https://www.bowlofdelicious.com/www.bowlofdelicious.com#subscribe\Subscribe]8;;\                                                                                                       

]8;id=143255;https://www.bowlofdelicious.com/www.bowlofdelicious.com\×]8;;\                                                                                                                  

]8;id=75301;https://www.bowlofdelicious.com/www.bowlofdelicious.com#recipe\Jump to Recipe]8;;\ - ]8;id=776223;https://www.bowlofdelicious.com/wprm_print/pan-fried-sea-bass-with-lemon-garlic-herb-sauce\Print]8;;\                                                                                             

4.98 from 655 votes                                                                                                

This Pan Fried Sea Bass with Lemon Garlic Herb Sauce is a 20 minute, one-pan, budget-friendly Mediterranean recipe 
that’s perfect on busy weeknights.                                                                                 

This Pan Fried Sea Bass recipe is a wonderful way to eat a mild, affordable* fish, and it’s packed with flavor from
fresh oregano, thyme, and parsley. It’s especially great if you have an abundance of fresh herbs in your garden you
need to use up! (raises hand)                                                                                      

\* Sea bass can vary greatly in price depending on variety, location, and if it’s in season. Barramundi is a       
reliably affordable option and is what I used.                                                                     

Originally, I was planning on doing a lemon caper sauce, similar to chicken piccata. But lo and behold, when I     
arrived at my refrigerator, I realized I was out of capers.                                                        

After some quick thinking and wandering around outside, I decided on a lemony, garlicky, fresh herb sauce instead. 

SO happy I was out of capers. This turned out wonderfully!                                                         


                                              How to Pan-Fry Sea Bass                                              

If you’ve ever made chicken piccata before, the process for this is very similar. First, pat the fish dry. It’s    
important that most of the moisture is removed from the fish, so it gets a nice brown color on the outside.        

Next dred

In [29]:
@weave.op 
@function_tool # <- we can use the decorator to automatically generate the tool schema
def exa_search(query: str, num_results: int = 5) -> list[dict[str, str]]:
    """Perform a search query on the web and retrieve the most relevant URLs and web content.
    
    This function uses the Exa search API to find relevant web pages based on the query
    and returns their titles, text content, and URLs.
    
    Args:
        query: The search query. Use detailed, specific queries for better results.
               The quality of results depends on the specificity of the query.
        num_results: The number of search results to retrieve. Defaults to 5.
    
    Returns:
        A list of dictionaries, each containing:
            - title: The title of the web page
            - text: The text content of the web page
            - url: The URL of the web page
    """
    search_results = exa_client.search_and_contents(query=query, type='auto', num_results=num_results)
    
    output = []
    for result in search_results.results:
        output.append(
            {"title": result.title,
            "text": result.text,
            "url": result.url
            }
        )
    return output

    

In [30]:
exa_search.tool_schema

{'type': 'function',
 'function': {'name': 'exa_search',
  'description': 'Perform a search query on the web and retrieve the most relevant URLs and web content.\n\nThis function uses the Exa search API to find relevant web pages based on the query\nand returns their titles, text content, and URLs.\n\nArgs:\n    query: The search query. Use detailed, specific queries for better results.\n           The quality of results depends on the specificity of the query.\n    num_results: The number of search results to retrieve. Defaults to 5.\n\nReturns:\n    A list of dictionaries, each containing:\n        - title: The title of the web page\n        - text: The text content of the web page\n        - url: The URL of the web page',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string', 'description': ''},
    'num_results': {'type': 'integer', 'description': '', 'default': 5}},
   'required': ['query']}}}

We get a list of results with the relevant metadata.

In [31]:
search_results = exa_search("How do I cook seabass?")
console.print(search_results)

[
    {
        'title': 'The BEST way to Cook Chilean Sea Bass that takes it to the NEXT ...',
        'text': '# [_Facebook_](https://www.facebook.com/)\n\n| | |\n| --- | --- |\n| Email or phone | Password 
|\n| [Forgot account?](https://www.facebook.com/recover/initiate?lwv=100&ars=royal_blue_bar) |\n\n[Create new 
account](https://www.facebook.com/r.php?locale=en_US)\n\n## You’re Temporarily Blocked\n\n## You’re Temporarily 
Blocked\n\nIt looks like you were misusing this feature by going too fast. You’ve been temporarily blocked from 
using it.\n\n[Back](https://www.facebook.com/www.facebook.com)\n\n- English (US)\n- 
[Español](https://www.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-best-
way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- [Français 
(France)](https://es-la.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-bes
t-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[中文(简体)](https://fr-fr.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-
best-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[العربية](https://zh-cn.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-bes
t-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- [Português 
(Brasil)](https://ar-ar.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-bes
t-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[Italiano](https://pt-br.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-be
st-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[한국어](https://it-it.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-best
-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[Deutsch](https://ko-kr.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-bes
t-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[हिन्दी](https://de-de.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-best-wa
y-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n- 
[日本語](https://hi-in.facebook.com/login/?next=https%3A%2F%2Fwww.facebook.com%2Fchefbillyparisi%2Fposts%2Fthe-best
-way-to-cook-chilean-sea-bass-that-takes-it-to-the-next-level%2F1239156604237561%2F)\n\n- [Sign 
Up](https://www.facebook.com/reg/)\n- [Log In](https://www.facebook.com/login/)\n- 
[Messenger](https://messenger.com/)\n- [Facebook Lite](https://www.facebook.com/lite/)\n- 
[Video](https://www.facebook.com/watch/)\n- [Meta Pay](https://about.meta.com/technologies/meta-pay)\n- [Meta 
Store](https://www.meta.com/)\n- [Meta Quest](https://www.meta.com/quest/)\n- [Ray-Ban 
Meta](https://www.meta.com/smart-glasses/)\n- [Meta AI](https://www.meta.ai/)\n- [Meta AI more 
content](https://www.meta.ai/pages/what-is-labubu/?utm_source=foa_web_footer)\n- 
[Instagram](https://l.facebook.com/l.php?u=https%3A%2F%2Fwww.instagram.com%2F&h=AT0-jEAtr7zT126tLWKVRA4hvnWz4eo9c9M
a3SnO8b6Kcj5yAY6G-X4XWuEItQo8e9xxX2xa-WUiz7YohLnNWuIJHWsqUOCFd44idFwg3em1Is04uCqQg-4LBgQh5yokgLWyzzY6_nY8xhMdse1Z0H
pCQ1dl7q29O8RkjQ)\n- [Threads](https://www.threads.com/)\n- [Voting Information 
Center](https://www.facebook.com/votinginformationcenter/?entry_point=c2l0ZQ%3D%3D)\n- [Privacy 
Policy](https://www.facebook.com/privacy/policy/?entry_point=facebook_page_footer)\n- [Consumer Health 
Privacy](https://www.facebook.com/privacy/policies/health/?entry_point=facebook_page_footer)\n- [Privacy 
Center](https://www.facebook.com/privacy/center/?entry_point=facebook_page_footer)\n- 
[About](https://about.meta.com/)

In [32]:
messages = [
    {"role": "system", "content": "You are an agent that has access to an advanced search engine. Please provide the user with the information they are looking for by using the search tool provided. Make sure to keep the sources. Return in markdown format."},
    {"role": "user", "content": "How do I cook seabass?"}]

response = call_model(model_name=MODEL_SMALL, messages=messages, tools=[exa_search.tool_schema])
console.print(response)

ChatCompletionMessage(
    content=None,
    refusal=None,
    role='assistant',
    annotations=None,
    audio=None,
    function_call=None,
    tool_calls=[
        ChatCompletionMessageFunctionToolCall(
            id='chatcmpl-tool-7326d249428a45da8b1d3b92441d162c',
            function=Function(
                arguments='{"query": "how to cook seabass step by step", "num_results": 5}',
                name='exa_search'
            ),
            type='function'
        )
    ],
    reasoning_content=None
)

Let's create some helper functions to perform the tool calls

In [33]:
def get_tool(tools: list[Callable], name: str) -> Callable:
    for t in tools:
        if t.__name__ == name:
            return t
    raise KeyError(f"No tool with name {name} found")

def perform_tool_calls(tools: list[Callable], tool_calls: list[Callable]) -> list[dict]:
    "Perform the tool calls and return the messages with the tool call results"
    messages = []
    if not tool_calls:
        return messages
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        with console.status(f"[bold cyan]Executing {function_name}...[/bold cyan]"):
            tool = get_tool(tools, function_name)
            tool_response = tool(**function_args) # doesn't handle async
        
        # Create panel content
        panel_content = f"[bold cyan]🔧 Tool Call:[/bold cyan] {function_name}\n\n"
        panel_content += f"[dim]Args: {tool_call.function.arguments}[/dim]\n\n"
        
        if isinstance(tool_response, list):
            panel_content += f"[green]✓[/green] Found {len(tool_response)} results"
        else:
            panel_content += f"[green]✓[/green] {function_name} executed successfully"
        
        console.print(Panel(panel_content, border_style="cyan"))
        
        messages.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "content": str(tool_response),
        })
    return messages

In [34]:

# add the tool call result to the messages
messages.append(response.model_dump())
messages.extend(perform_tool_calls(tools=[exa_search], tool_calls=response.tool_calls))
messages.append({
    "role": "user",
    "content": "Answer my previous query based on the search results.",})

final_response = call_model(model_name=MODEL_SMALL, messages=messages)
console.rule("Final Model Response")
console.md(final_response.content)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 Tool Call: exa_search                                                                                        │
│                                                                                                                 │
│ Args: {"query": "how to cook seabass step by step", "num_results": 5}                                           │
│                                                                                                                 │
│ ✓ Found 5 results                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────── Final Model Response ───────────────────────────────────────────────

Cooking seabass—especially Chilean sea bass or branzino—can yield restaurant-quality results with simple           
techniques. Here are several reliable methods based on top-rated recipes:                                          

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                       1. Pan-Seared Chilean Sea Bass with Lemon Butter Sauce (Alphafoodie)                        

Prep: 5 min | Cook: 10 min | Total: 15 min                                                                         

Ingredients:                                                                                                       

 • 4 skin-on Chilean sea bass fillets (5–6 oz each)                                                                
 • 1.5 tbsp olive oil + 1.5 tbsp butter (or avocado oil/ghee)                                                      
 • Salt & black pepper                                                                                             
 • Lemon butter sauce: ¼ cup butter, 2 tbsp lemon juice, salt, pepper (optional: garlic)                           

Steps:                                                                                                             

 1 Pat fillets dry and season both sides with salt and pepper.                                                     
 2 Heat oil and butter in a skillet over medium-high heat.                                                         
 3 Place fillets skin-side down and sear 3–4 minutes until crispy. Flip and cook 3–4 minutes more.                 
 4 Check doneness: internal temperature should be 140–145°F (60–63°C) or flake easily with a fork.                 
 5 Meanwhile, melt butter for sauce, stir in lemon juice, salt, and pepper.                                        
 6 Spoon sauce over fish and serve with sides like mashed potatoes, quinoa, or sautéed greens.                     

Tip: Don’t move the fish until it releases easily from the pan to ensure a crispy crust.                           

👉 ]8;id=930794;https://www.alphafoodie.com/chilean-sea-bass-recipe/\Source: Alphafoodie]8;;\                                                                                             

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
               2. Mediterranean Pan-Seared Sea Bass with Bell Pepper Medley (The Mediterranean Dish)               

Prep: 5 min | Cook: 10 min | Total: 15–

Let's wrap this in a function:

In [35]:
@weave.op
def research(query: str) -> str:
    messages = [
        {"role": "system", "content": "You are an agent that has access to an advanced search engine. Please provide the user with the information they are looking for by using the search tool provided. Make sure to keep the sources. Return in markdown format."},
        {"role": "user", "content": query}]

    # call model with tools
    response = call_model(
        model_name=MODEL_SMALL, 
        messages=messages, 
        tools=[exa_search.tool_schema])

    # add the response to the messages
    messages.append(response.model_dump())

    # perform the tool calls
    messages.extend(perform_tool_calls(tools=[exa_search], tool_calls=response.tool_calls))
    
    # prompt the model to be grounded
    messages.append({"role": "user","content": "Answer my previous query based on the search results.",})

    final_response = call_model(model_name=MODEL_SMALL, messages=messages)
    return final_response.content

In [36]:
result = research("What are the most popular pokemons?")
console.md(result)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 Tool Call: exa_search                                                                                        │
│                                                                                                                 │
│ Args: {"query": "most popular Pokémon of all time", "num_results": 5}                                           │
│                                                                                                                 │
│ ✓ Found 5 results                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Based on the search results, the most popular Pokémon vary depending on the poll and region, but several           
consistently stand out:                                                                                            

 1 Umbreon – According to a massive fan poll by YouTuber Thomas Game Docs involving 1.6 million votes, Umbreon     
   ranked first with an impressive 79.5% preference, making it the top favorite among fans in that survey.         
 2 Charizard – Multiple sources highlight Charizard as one of the most beloved Pokémon. According to a Guide Strats
   analysis of over 75,000 fan profiles, Charizard is the most popular Pokémon overall, with 3,105                 
   favorites—outpacing even Pikachu. IGN also ranked Charizard as the #1 Pokémon, praising its iconic status and   
   role in the anime.                                                                                              
 3 Pikachu – While Pikachu is the official mascot of the franchise, it ranked lower in some fan polls (65th in the 
   Thomas Game Docs poll), but remains a cultural icon and is especially popular in Canada.                        
 4 Eevee and its Evolutions – Eevee and its evolutions (like Sylveon and Vaporeon) are extremely popular. Eevee    
   ranked 4th in the Thomas Game Docs poll and is especially loved for its versatility and cuteness.               
 5 Other Fan Favorites – According to a 2025 Japanese fan poll focused on Pokémon Legends: Z-A, the top 10 most    
   popular Pokémon included:                                                                                       
    • Charizard                                                                                                    
    • Lucario                                                                                                      
    • Greninja                                                                                                     
    • Chandelure                                                                                                   
    • Absol                                                                                                        
    • Gardevoir                                                                                                    

                                         Summary of Most Popular Pokémon:                                          

                                             
  Pokémon   Notable Rankings                 
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Umbreon   #1 in 1.6M-vote global fan poll

![](images/04_pokedex.png)

This is "Almost" an agent, but it's missing the loop. Let's add that next.

## 4. Implementing a Basic Agentic Loop

Let's implement a basic agentic loop. We'll use the `pokedex` function we just created. The implementation we have above has some limitations:
- Its a single turn, so if it fails to answer my question in one pass it is over.

![](images/05_agent.png)

From the really good [Anthropic Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) article and encourage people to read it.

A simple for loop

In [37]:
@weave.op
def research_loop(query: str, max_turns: int = 4, tools = [exa_search, add_numbers]) -> str:
    messages = [
        {"role": "system", "content": "You are an agent that has access to an advanced search engine. Please provide the user with the information they are looking for by using the search tool provided. Make sure to keep the sources. Always use tools to obtain reliable results. Return the final answer in markdown format."},
        {"role": "user", "content": query}]
    
    for turn in range(max_turns):
        console.rule(f"Agent Loop Turn {turn + 1}/{max_turns}")

        # call model with tools
        response = call_model(
            model_name=MODEL_MEDIUM, 
            messages=messages, 
            tools=[t.tool_schema for t in tools])

        # add the response to the messages
        messages.append(response.model_dump())

        # if the LLM requested tool calls, perform them
        if response.tool_calls:
            # perform the tool calls
            tool_outputs = perform_tool_calls(tools=tools, tool_calls=response.tool_calls)
            messages.extend(tool_outputs)
        # LLM gave content response
        elif response.content:
            console.rule("Final Model Response")
            console.md(response.content)
            return response.content
        else:
            print("LLM response had neither content nor tool calls. Stopping loop.")
            break

In [38]:
res = research_loop("What is the sum of the populations of the 2 major EU cities?")

─────────────────────────────────────────────── Agent Loop Turn 1/4 ───────────────────────────────────────────────

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 Tool Call: exa_search                                                                                        │
│                                                                                                                 │
│ Args: {"query": "largest most populous EU cities population European Union 2024", "num_results": 5}             │
│                                                                                                                 │
│ ✓ Found 5 results                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Agent Loop Turn 2/4 ───────────────────────────────────────────────

────────────────────────────────────────────── Final Model Response ───────────────────────────────────────────────

Based on the search results, I can identify the 2 most populous cities in the European Union and calculate their   
combined population.                                                                                               

According to the Wikipedia page "List of cities in the European Union by population within city limits", the two   
largest EU cities are:                                                                                             

 1 Berlin, Germany: 3,685,265 people (as of December 31, 2024)                                                     
 2 Madrid, Spain: 3,506,730

# 5. Structuring the Agent with Classes

The loop above works, but for more complex agents, encapsulating the logic and state within classes is much better. We'll define:
- `AgentState`: A Pydantic model to hold the conversation history and potentially other state.
- `SimpleAgent`: A class containing the agent's configuration (model, system message, tools) and logic (`step`, `run`).

In [39]:
class AgentState(BaseModel):
    """Manages the state of the agent."""
    messages: list[dict[str, Any]] = Field(default_factory=list)
    step: int = Field(default=0)
    final_assistant_content: str | None = None # Populated at the end of a run

In [40]:
class SimpleAgent:
    """A simple agent class with tracing, state, and tool processing."""
    def __init__(self, model_name: str, system_message: str, tools: list[Callable], state_class: BaseModel = AgentState):
        self.model_name = model_name
        self.system_message = system_message
        self.tools = [function_tool(t) for t in tools] # add schemas to the tools
        self.state_class = state_class
    
    @weave.op(name="SimpleAgent.step") # Trace each step
    def step(self, state: AgentState) -> AgentState:
        step = state.step + 1
        messages = state.messages
        final_assistant_content = None
        try:
            # call model with tools
            response = call_model(
                model_name=self.model_name, 
                messages=messages, 
                tools=[t.tool_schema for t in self.tools])

            # add the response to the messages
            messages.append(response.model_dump())

            # if the LLM requested tool calls, perform them
            if response.tool_calls:
                # perform the tool calls
                tool_outputs = perform_tool_calls(tools=self.tools, tool_calls=response.tool_calls)
                messages.extend(tool_outputs)

            # LLM gave content response
            else:
                final_assistant_content = response.content
        except Exception as e:
            console.print(f"ERROR in Agent Step: {e}")
            # Add an error message to history to indicate failure
            messages.append({"role": "assistant", "content": f"Agent error in step: {str(e)}"})
            final_assistant_content = f"Agent error in step {step}: {str(e)}"
        return self.state_class(messages=messages, step=step, final_assistant_content=final_assistant_content)

    @weave.op(name="SimpleAgent.run")
    def run(self, user_prompt: str, max_turns: int = 10, **state_kwargs: Any) -> AgentState:
        state = self.state_class(messages=[
            {"role": "system", "content": self.system_message},
            {"role": "user", "content": user_prompt}], **state_kwargs)
        for _ in range(max_turns):
            console.rule(f"Agent Loop Turn {state.step+1}/{max_turns}")
            state = self.step(state)
            if state.final_assistant_content:
                return state
        return state



![](images/07_simple_agent.png)

In [41]:
agent = SimpleAgent(
    model_name=MODEL_SMALL,
    system_message="You are an agent that has access to an advanced search engine. Please provide the user with the information they are looking for by using the search tool provided. Make sure to keep the sources. Always use tools to obtain reliable results. Return the final answer in markdown format.",
    tools=[exa_search, add_numbers],
)
state = agent.run(user_prompt="What is the combined weight of Ash's first 3 pokemons?", max_turns=3)
print(f"Final response: {state.final_assistant_content}")


─────────────────────────────────────────────── Agent Loop Turn 1/3 ───────────────────────────────────────────────

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 Tool Call: exa_search                                                                                        │
│                                                                                                                 │
│ Args: {"query": "Ash's first three Pokémon weights", "num_results": 5}                                          │
│                                                                                                                 │
│ ✓ Found 5 results                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Agent Loop Turn 2/3 ───────────────────────────────────────────────

ERROR in Agent Step: Error code: 400 - {'error': {'message': "This model's maximum context length is 262144 tokens.
However, your request has 326960 input tokens. Please reduce the length of the input messages. None", 'type': 
'BadRequestError', 'param': None, 'code': 400}}

Final response: Agent error in step 2: Error code: 400 - {'error': {'message': "This model's maximum context length is 262144 tokens. However, your request has 326960 input tokens. Please reduce the length of the input messages. None", 'type': 'BadRequestError', 'param': None, 'code': 400}}


In [44]:
@weave.op 
@function_tool # <- we can use the decorator to automatically generate the tool schema
def exa_search_and_refine(query: str, num_results: int = 5) -> list[dict[str, str]]:
    """Perform a search query on the web and retrieve the most relevant URLs and web content.
    
    This function uses the Exa search API to find relevant web pages based on the query
    and returns their titles, text content, and URLs.
    
    Args:
        query: The search query. Use detailed, specific queries for better results.
               The quality of results depends on the specificity of the query.
        num_results: The number of search results to retrieve. Defaults to 5.
    
    Returns:
        A list of dictionaries, each containing:
            - title: The title of the web page
            - text: The text content of the web page
            - url: The URL of the web page
    """
    search_results = exa_client.search_and_contents(query=query, type='auto', num_results=num_results)
    
    @weave.op
    def refine_search_result(result, query):
        messages = [
            {"role":"system", "content": f"Your task is to extract from the search results only the info that is relevant to answer the query"},
            {"role": "user", "content": f"- query: {query}\n- Search result: {result}"}
        ]
        refined_search = call_model(model_name=MODEL_SMALL, messages=messages)
        return refined_search.content

    output = []
    for item, result in enumerate(search_results.results):
        console.print(f"Refining result {item+1}")
        refined_text = refine_search_result(result.text, query)
        output.append(
            {"title": result.title,
            "text": refined_text,
            "url": result.url
            }
        )
    return output

In [45]:
agent = SimpleAgent(
    model_name=MODEL_SMALL,
    system_message="You are an agent that has access to an advanced search engine. Please provide the user with the information they are looking for by using the search tool provided. Make sure to keep the sources. Always use tools to obtain reliable results. Return the final answer in markdown format.",
    tools=[exa_search_and_refine, add_numbers]
)
state = agent.run(user_prompt="What is the combined weight of Ash's first 3 pokemons?", max_turns=3)
print(f"Final response: {state.final_assistant_content}")

─────────────────────────────────────────────── Agent Loop Turn 1/3 ───────────────────────────────────────────────

Refining result 1

Refining result 2

Refining result 3

Refining result 4

Refining result 5

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 Tool Call: exa_search_and_refine                                                                             │
│                                                                                                                 │
│ Args: {"query": "Ash's first three Pokémon names and weights", "num_results": 5}                                │
│                                                                                                                 │
│ ✓ Found 5 results                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Agent Loop Turn 2/3 ───────────────────────────────────────────────

Final response: Ash's first three Pokémon are generally recognized as **Pikachu**, **Bulbasaur**, and **Charmander** (who later evolved into **Charizard**). However, some sources consider **Caterpie** and **Pidgeotto** as part of his initial team since they were among the first Pokémon he caught.

To clarify:

- **Pikachu** – 13.2 lbs (6.0 kg)  
- **Bulbasaur** – 15.2 lbs (6.9 kg)  
- **Charizard** – 199.5 lbs (90.5 kg)  

> Note: Charmander evolves into Charizard, and since Ash’s Charmander fully evolved, using Charizard’s weight is appropriate.

### Combined Weight:
$$
13.2 \, \text{lbs} + 15.2 \, \text{lbs} + 199.5 \, \text{lbs} = 227.9 \, \text{lbs}
$$

$$
6.0 \, \text{kg} + 6.9 \, \text{kg} + 90.5 \, \text{kg} = 103.4 \, \text{kg}
$$

### ✅ Final Answer:
The combined weight of Ash's first three Pokémon (Pikachu, Bulbasaur, and Charizard) is **227.9 pounds (103.4 kilograms)**.

**Sources:**
- [Bulbapedia - Pokémon by weight](https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9

Possible improvements to the SimpleAgent:
- Give the model info about the state of the conversation, you could inject a message with the model context pressure, steps left, etc.
- Structured output. Make the model output a specific format, for instance a JSON with the expected fields.
- Add more tools like read and write files, access a database.
- Agent handoff: Agent1 does triage and Agent2 executes specific tasks.

## Evaluating our SimpleAgent

In [ ]:
# Run this in colab

# !git clone https://github.com/morganmcg1/deep-research-bot/
# !mv deep-research-bot/data ./data

In [57]:
import sys
from functools import partial
from pathlib import Path

# Add project root to Python path
# Handle both cases: running from notebooks/ or from project root
cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd  # Already at project root

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [58]:
from pathlib import Path
from deep_research_bot.evaluation.eval import run_evaluation
from deep_research_bot.evaluation.eval_config import EvalConfig

MAX_TURNS=5

agent = SimpleAgent(
    model_name=MODEL_SMALL,
    system_message="You are an agent that has access to an advanced search engine. Please provide the user with the information they are looking for by using the search tool provided. Make sure to keep the sources. Always use tools to obtain reliable results. Return the final answer in markdown format.",
    tools=[exa_search_and_refine, add_numbers],
)

eval_config = EvalConfig(
    evaluation_name=f"SimpleAgent_max-turns-{MAX_TURNS}_{agent.model_name.split('/')[-1]}",
    trials=1,
    limit=4,
    judge_model="deepseek-ai/DeepSeek-R1-0528",
    weave_parallelism=4,
    queries=project_root / "data/prompt_data/query.jsonl",
    reference=project_root / "data/test_data/cleaned_data/reference.jsonl",
    criteria=project_root / "data/criteria_data/criteria.jsonl",
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
)

results = await run_evaluation(
    eval_config=eval_config,
    agent_callable=partial(agent.run, max_turns=MAX_TURNS),  # <- partial to limit the number of agent turns
)
results

4 dataset rows constructed for evaluation


─────────────────────────────────────────────── Agent Loop Turn 1/5 ───────────────────────────────────────────────

─────────────────────────────────────────────── Agent Loop Turn 1/5 ───────────────────────────────────────────────

─────────────────────────────────────────────── Agent Loop Turn 1/5 ───────────────────────────────────────────────

─────────────────────────────────────────────── Agent Loop Turn 1/5 ───────────────────────────────────────────────

...

Refining result 5

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 Tool Call: exa_search_and_refine                                                                             │
│                                                                                                                 │
│ Args: {"query": "hybrid models combining Mean-Variance, Black-Litterman, and deep learning for robust asset     │
│ allocation in FinTech", "num_results": 5}                                                                       │
│                                                                                                                 │
│ ✓ Found 5 results                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Agent Loop Turn 5/5 ───────────────────────────────────────────────

weave: Evaluated 2 of 4 examples
INFO:weave.evaluation.eval:Evaluated 2 of 4 examples
weave: Evaluated 3 of 4 examples
INFO:weave.evaluation.eval:Evaluated 3 of 4 examples
weave: Evaluated 4 of 4 examples
INFO:weave.evaluation.eval:Evaluated 4 of 4 examples
weave: Evaluation summary {
weave:   "output": {
weave:     "row_index": {
weave:       "mean": 1.5
weave:     },
weave:     "trial_index": {
weave:       "mean": 0.0
weave:     },
weave:     "id": {
weave:       "mean": 52.5
weave:     }
weave:   },
weave:   "deep_research_scores": {
weave:     "comprehensiveness": 0.2075126226445868,
weave:     "insight": 0.18385863199108723,
weave:     "instruction_following": 0.2940417908374713,
weave:     "readability": 0.3704657611815788,
weave:     "overall": 0.24706609368681345
weave:   },
weave:   "model_latency": {
weave:     "mean": 84.5908635854721
weave:   }
weave: }
INFO:weave.evaluation.eval:Evaluation summary {
  "output": {
    "row_index": {
      "mean": 1.5
    },
    "trial_inde

{'output': {'row_index': {'mean': 1.5},
  'trial_index': {'mean': 0.0},
  'id': {'mean': 52.5}},
 'deep_research_scores': {'comprehensiveness': 0.2075126226445868,
  'insight': 0.18385863199108723,
  'instruction_following': 0.2940417908374713,
  'readability': 0.3704657611815788,
  'overall': 0.24706609368681345},
 'model_latency': {'mean': 84.5908635854721}}